# Système intelligent de matching CV ↔ offres d'emploi
### Recommandation du Top 3 des offres et explications fiables

| | |
|---|---|
| **Établissement** | Université Internationale de Rabat (UIR) — Intelligence Artificielle |
| **Auteurs** | *Prénom NOM 1* · *Prénom NOM 2* |
| **Encadrant** | *Pr. NOM* |
| **Date** | *JJ/MM/AAAA* |

---

**Ce notebook constitue le rendu complet du projet.** Il présente la problématique, les données, la méthode,
le code réellement utilisé, le protocole d'évaluation, les résultats mesurés, les aspects éthiques et les limites.
Tout le code exécuté provient du package `cvmatch` (dossier `src/`) : les chiffres affichés sont **calculés par
le notebook**, aucun n'est saisi à la main.

**Sommaire**
1. Introduction et objectifs
2. Environnement et reproductibilité
3. Données : catalogue d'offres Rekrute (analyse exploratoire)
4. Prétraitement : PDF, vie privée, nettoyage, compétences
5. Modèles de classement : TF-IDF (baseline) et Sentence-BERT
6. Explications : règles et Gemini avec garde-fou anti-invention
7. Protocole d'évaluation (vérité terrain)
8. Résultats et ablations
9. Démonstration du système complet
10. Éthique et vie privée
11. Limites et perspectives
12. Conclusion
13. Annexes (architecture, reproduction, traçabilité)

## 1. Introduction et objectifs

**Problématique.** Un étudiant ou un jeune diplômé passe beaucoup de temps à lire des offres d'emploi et à
comparer manuellement leurs exigences avec son CV. Il lui est difficile de savoir quelles offres correspondent le
mieux à son profil, et **pourquoi**.

**Objectif.** À partir d'un CV au format PDF, le système :
1. extrait le texte et **masque les données personnelles** ;
2. recommande les **3 offres les plus proches** dans un catalogue réel d'offres marocaines (Rekrute) ;
3. **explique** chaque recommandation : compétences en commun, compétences à développer, extraits justificatifs.

**Questions de recherche (évaluées par ablation).**
- *Q1* — Un modèle sémantique (Sentence-BERT) classe-t-il mieux qu'une baseline lexicale (TF-IDF) ?
- *Q2* — Le nettoyage du texte améliore-t-il le classement ?
- *Q3* — Un LLM (Gemini) produit-il des explications fidèles aux sources, et la description de l'offre est-elle
  nécessaire ?

> Le score de matching est une **proximité textuelle/sémantique**, pas une probabilité d'embauche.

**Vue d'ensemble du pipeline**

```
 CV (PDF) ─► extraction texte ─► masquage PII ─► nettoyage ─► classement (TF-IDF | SBERT) ─► Top 3
                                                                                              │
                                                     explications (règles | Gemini + garde-fou) ◄┘

 Hors-ligne : prepare → embed → cvs → pool → labels → ablate-ml → ablate-llm → report
```

## 2. Environnement et reproductibilité

La cellule suivante se place à la racine du projet. Sur **Google Colab**, elle demande le zip du projet
(créé avec `cvmatch package`) et l'installe.

**Options d'exécution** (cellule d'après) :
- `RUN_PIPELINE = True` : exécute le pipeline hors-ligne s'il manque des sorties (long la première fois :
  téléchargement du dataset et calcul des embeddings, ≈ 45 min sur CPU pour 15 000 offres).
- `OFFLINE = True` : démonstration sans Internet (offres fictives + encodeur factice) — **chiffres sans valeur
  scientifique**, uniquement pour vérifier que tout s'exécute.

In [ ]:
import os, sys, zipfile, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    if not IN_COLAB:
        raise SystemExit("Ouvrez ce notebook depuis le dossier du projet (celui qui contient pyproject.toml).")
    from google.colab import files                       # Colab : téléverser cv-matching.zip
    name = next(iter(files.upload()))
    zipfile.ZipFile(name).extractall("/content")
    ROOT = next(p for p in Path("/content").iterdir() if (p / "pyproject.toml").exists())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[ml,llm,dev]"], check=True)

os.chdir(ROOT)
os.environ["CVMATCH_HOME"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Racine du projet :", ROOT)

In [ ]:
RUN_PIPELINE = True     # exécute les étapes manquantes du pipeline
OFFLINE = False         # True = démonstration sans réseau (chiffres NON rapportables)

import warnings, logging
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown

import cvmatch
from cvmatch.config import get_settings
from cvmatch.utils.logging import setup_logging
setup_logging("WARNING")
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

S = get_settings()
if OFFLINE:
    S.sbert_model = "hashing"
    os.environ["SBERT_MODEL"] = "hashing"
print(f"cvmatch {cvmatch.__version__} | Python {sys.version.split()[0]} | pandas {pd.__version__}")
print(f"Modèle SBERT : {S.sbert_model} | Gemini configuré : {bool(S.gemini_api_key)}")

In [ ]:
# Exécution du pipeline (seules les étapes dont les sorties manquent sont recalculées)
from cvmatch.pipeline import PipelineConfig, run_pipeline

CFG = PipelineConfig.load(ROOT / "configs" / "pipeline.yaml")
if OFFLINE:
    CFG.data.source = "synthetic"
needed = [S.jobs_path, S.results_dir / "report.md"]
if RUN_PIPELINE and not all(p.exists() for p in needed):
    setup_logging("INFO")
    run_pipeline(CFG, S)
    setup_logging("WARNING")
else:
    print("Sorties du pipeline déjà présentes : rien à recalculer.")

## 3. Données : catalogue d'offres Rekrute

**Source.** Jeu de données public `adlbh/rekrute-2005-2022` (Hugging Face) : offres publiées sur Rekrute.com,
principal site d'emploi marocain, de 2005 à 2022.

**Préparation (`cvmatch prepare`, module `data/catalog.py`).**
- conservation des offres **récentes** (≥ 2019 par défaut) ;
- construction d'une description à partir des champs *Poste*, *Profil recherché*, *Qualités requises*, *soft skills* ;
- suppression des offres sans identifiant, sans titre ou à description trop courte (< 80 caractères) ;
- **dédoublonnage** exact (même titre + même description normalisée), puis échantillonnage reproductible
  (`seed = 42`) ;
- validation : identifiants uniques, aucune description vide.

In [ ]:
from cvmatch.data import load_jobs
jobs = load_jobs(S.jobs_path)
print(f"{len(jobs):,} offres après préparation | colonnes : {list(jobs.columns)}")
display(jobs[["job_id", "job_title", "company", "city", "sector", "year"]].head(5))

In [ ]:
desc_len = jobs["job_description"].str.split().str.len()
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
jobs["year"].value_counts().sort_index().plot.bar(ax=ax[0], color="#4C72B0", title="Offres par année")
(jobs["sector"].replace("", "Non renseigné").str.split("/").str[0].str.strip()
     .value_counts().head(10).sort_values().plot.barh(ax=ax[1], color="#55A868", title="Top 10 secteurs"))
desc_len.clip(upper=desc_len.quantile(0.99)).plot.hist(bins=40, ax=ax[2], color="#C44E52",
                                                      title="Longueur des descriptions (mots)")
plt.tight_layout(); plt.show()
display(pd.DataFrame({
    "villes distinctes": [jobs["city"].nunique()], "entreprises distinctes": [jobs["company"].nunique()],
    "mots / description (médiane)": [int(desc_len.median())],
    "mots / description (max)": [int(desc_len.max())],
}))

In [ ]:
# Compétences les plus demandées (dictionnaire FR/EN du projet, échantillon de 3 000 offres)
from cvmatch.preprocessing import extract_skills
sample = jobs.sample(min(3000, len(jobs)), random_state=0)
skills = pd.Series([s for t, d in zip(sample.job_title, sample.job_description)
                    for s in extract_skills(f"{t}. {d}")])
top = (skills.value_counts().head(20) / len(sample) * 100).sort_values()
top.plot.barh(figsize=(8, 6), color="#8172B2",
              title="Compétences les plus citées (% des offres)"); plt.xlabel("%"); plt.show()

## 4. Prétraitement

| Étape | Module | Rôle |
|---|---|---|
| Extraction PDF | `preprocessing/pdf.py` | texte via PyMuPDF ; rejet des PDF scannés (sans texte) |
| Masquage PII | `preprocessing/pii.py` | e-mail, téléphones (formats marocains et internationaux), liens LinkedIn/GitHub, adresse |
| Nettoyage | `preprocessing/cleaning.py` | HTML, URL, puces, caractères parasites — **en préservant** `C++`, `C#`, `.NET`, `Node.js` |
| Compétences | `preprocessing/skills.py` | dictionnaire FR/EN (~100 compétences, alias, insensible aux accents) |

Le **masquage est appliqué avant tout** : c'est le texte masqué qui est journalisé et éventuellement envoyé à
Gemini.

In [ ]:
from cvmatch.preprocessing import clean_text, mask_pii
from cvmatch.evaluation.annotation import load_cvs

CVS = load_cvs(S.cvs_dir)
cv_id = sorted(CVS)[0]
raw_cv = CVS[cv_id]
print(f"{len(CVS)} CV de test chargés. Exemple : {cv_id}\n")
print("── CV brut (extrait) ──────────────────────────\n" + raw_cv[:420])
print("\n── Après masquage PII ─────────────────────────\n" + mask_pii(raw_cv)[:420])

In [ ]:
exemple = "<p>Expert C++, C# et .NET ; Node.js &amp; SQL</p> • voir https://exemple.com"
print("Avant :", exemple)
print("Après :", clean_text(exemple))
print("\nCompétences détectées dans le CV :", extract_skills(mask_pii(raw_cv)))

## 5. Modèles de classement

Les deux modèles partagent la **même interface** (`rank(texte, top_k, candidates)`), ce qui permet de ne changer
qu'**un seul facteur** par expérience.

**B0 — TF-IDF + cosinus (baseline).** Unigrammes et bigrammes, `sublinear_tf`, suppression des accents, tokenisation
conservant les termes techniques. Rapide, interprétable, mais purement lexical : *« analyste de données »* et
*« data analyst »* ne se rapprochent pas.

**F1 — Sentence-BERT.** Modèle multilingue `paraphrase-multilingual-MiniLM-L12-v2` (FR/EN/AR). Les embeddings des
15 000 offres sont **pré-calculés une fois** (`cvmatch embed`) ; à la requête, seul le CV est encodé. Un CV long est
découpé en blocs de 110 mots dont les vecteurs sont moyennés (le modèle tronque ~128 tokens). Score = cosinus.

**Diversité du Top 3.** Une même offre est souvent publiée plusieurs fois (agences, rééditions) : `diversify`
écarte les intitulés quasi identiques.

$$\text{score}(cv, o) = \cos(\vec{cv}, \vec{o}) = \frac{\vec{cv}\cdot\vec{o}}{\lVert\vec{cv}\rVert\,\lVert\vec{o}\rVert}$$

In [ ]:
import inspect
from cvmatch.ranking import TfidfRanker, SbertRanker, diversify, build_ranker
print(inspect.getsource(TfidfRanker))
print(inspect.getsource(SbertRanker.encode_resume))

In [ ]:
from cvmatch.preprocessing import prepare_resume

masked, query = prepare_resume(raw_cv, cleaning=True)
rankers = {"TF-IDF (B0)": build_ranker(jobs, S, name="tfidf")}
try:
    rankers["Sentence-BERT (F1)"] = build_ranker(jobs, S, name="sbert")
except Exception as exc:
    print("Sentence-BERT indisponible :", exc)

titles = jobs["job_title"].tolist()
for name, rk in rankers.items():
    top = diversify(rk.rank(query, top_k=15), titles, 3)
    display(Markdown(f"**{name}** — Top 3 pour `{cv_id}`"))
    display(pd.DataFrame([{"score": round(r.score, 3), "offre": jobs.iloc[r.index].job_title,
                           "entreprise": jobs.iloc[r.index].company, "ville": jobs.iloc[r.index].city}
                          for r in top]))

## 6. Explications des recommandations

**L0 — Explication à règles (sans LLM).** Intersection entre les compétences du CV et celles de l'offre (même
dictionnaire), compétences manquantes, et phrases-preuves extraites des deux textes. Toujours disponible, rapide,
entièrement vérifiable.

**L1 — Gemini (`gemini-2.5-flash`).** Prompt versionné imposant une sortie JSON, température 0, n'utilisant **que**
le CV masqué et l'offre. Cache disque, réessais avec backoff, et **repli automatique** sur L0 (clé absente, quota…).

**Garde-fou anti-invention.** Chaque élément renvoyé par Gemini est vérifié dans les sources :
- une compétence « en commun » doit figurer **dans le CV et dans l'offre** ;
- une compétence « manquante » ne doit **pas** figurer dans le CV ;
- une preuve doit être un extrait **réellement présent** dans le CV ou l'offre.

Ce qui échoue est retiré et compté (`guard_removed`) : c'est la mesure de fidélité utilisée en évaluation.

In [ ]:
from cvmatch.explain import RuleExplainer
best = diversify(rankers["TF-IDF (B0)"].rank(query, top_k=15), titles, 1)[0]
job = jobs.iloc[best.index][["job_title", "job_description"]].to_dict()
e = RuleExplainer("fr").explain(masked, job)
print("Offre :", job["job_title"])
print("Explication :", e.explanation)
print("En commun   :", e.matching_skills)
print("À développer:", e.missing_skills)
for ev in e.evidence:
    print("  •", ev)

In [ ]:
# Démonstration du garde-fou : on simule une réponse de LLM contenant des inventions
import json
from cvmatch.explain import GeminiExplainer

CV_DEMO = "Analyste. Je maîtrise Python et SQL. Excel au quotidien."
JOB_DEMO = {"job_title": "Data Analyst",
            "job_description": "Analyser des données avec Python et SQL. Maîtrise de Power BI."}
reponse_llm = {
    "matching_skills": ["Python", "SQL", "Kubernetes"],          # Kubernetes : inventé
    "missing_skills": ["Power BI", "Excel"],                     # Excel est dans le CV : contradiction
    "explanation": "Profil cohérent avec le poste.",
    "evidence": ["Je maîtrise Python et SQL", "5 ans chez Google"],  # 2e preuve : inventée
}

class LLMSimule:
    models = property(lambda self: self)
    def generate_content(self, model, contents, config):
        return type("R", (), {"text": json.dumps(reponse_llm)})()

e = GeminiExplainer("demo", client=LLMSimule(), min_interval=0).explain(CV_DEMO, JOB_DEMO)
display(pd.DataFrame({
    "champ": ["matching_skills", "missing_skills", "evidence", "éléments retirés"],
    "réponse LLM": [reponse_llm["matching_skills"], reponse_llm["missing_skills"], reponse_llm["evidence"], ""],
    "après garde-fou": [e.matching_skills, e.missing_skills, e.evidence,
                        f"{e.guard_removed} ({', '.join(e.unverified_skills)}, 1 preuve)"],
}))

In [ ]:
# Gemini réel (si GEMINI_API_KEY est définie dans .env ou dans les secrets Colab)
if IN_COLAB and not S.gemini_api_key:
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY"); S = get_settings()
    except Exception:
        pass
if S.gemini_api_key:
    from cvmatch.explain import build_explainers
    g = build_explainers(S)["gemini"].explain(masked, job, extract_skills(masked))
    print("source :", g.source, "| retirés par le garde-fou :", g.guard_removed)
    print(g.explanation); print("En commun :", g.matching_skills); print("À développer :", g.missing_skills)
else:
    print("Pas de clé Gemini : cette cellule est ignorée (le système utilise alors l'explication à règles).")

## 7. Protocole d'évaluation

**Pourquoi un pool ?** Annoter 15 000 offres par CV est impossible. Pour chaque CV de test, on construit un
**pool de candidats** : top-6 TF-IDF ∪ top-6 SBERT ∪ 4 offres aléatoires, présentées **dans un ordre mélangé**
(les annotateurs ne voient ni le modèle ni le rang).

**Annotation.** Deux membres annotent **indépendamment** chaque paire (CV, offre) : 1 = pertinente, 0 = non, selon une
grille écrite (métier visé, compétences clés compatibles, niveau d'études cohérent). Une offre est jugée pertinente
si **les deux** annotateurs ont mis 1. L'accord inter-annotateurs est mesuré par le **kappa de Cohen**.

**Évaluation sur pool figé.** Chaque configuration classe **uniquement** les offres annotées de chaque CV : une paire
non annotée n'est jamais comptée comme négative, et toutes les configurations voient exactement les mêmes candidats.

**Métrique.** $\text{Precision@3} = \frac{|\{\text{offres pertinentes dans le Top 3}\}|}{3}$, moyennée sur les CV,
ainsi que la latence médiane du classement.

**Labels proxy.** Sans annotation humaine, le pipeline utilise des labels automatiques par mots-clés : ils servent
**uniquement** à vérifier la chaîne technique et sont signalés `_PROXY`.

In [ ]:
from cvmatch.evaluation.annotation import has_annotations
pool_csv = S.eval_dir / "pool_to_annotate.csv"
pool = pd.read_csv(pool_csv)
ANNOTATED = has_annotations(pool_csv)
print(f"Pool : {len(pool)} paires (CV, offre) pour {pool.cv_id.nunique()} CV "
      f"(≈ {len(pool) / max(pool.cv_id.nunique(), 1):.1f} candidats par CV)")
if ANNOTATED:
    from sklearn.metrics import cohen_kappa_score
    a = pool.dropna(subset=["annotator_1", "annotator_2"])
    a1, a2 = a.annotator_1.astype(int), a.annotator_2.astype(int)
    display(pd.DataFrame([{"paires annotées": len(a), "accord brut": round((a1 == a2).mean(), 3),
                           "kappa de Cohen": round(cohen_kappa_score(a1, a2), 3),
                           "pertinentes (les deux = 1)": int(((a1 == 1) & (a2 == 1)).sum())}]))
else:
    display(Markdown("> ⚠ **Pool non annoté** : les résultats ci-dessous reposent sur des **labels proxy** "
                     "et ne doivent pas être rapportés. Annotez `data/eval/pool_to_annotate.csv` puis "
                     "relancez `cvmatch pipeline --from labels`."))
display(pool[["cv_id", "job_title", "company", "annotator_1", "annotator_2"]].head(6))

## 8. Résultats

### 8.1 Ablation du classement (Q1, Q2)

| Test | Configuration | Facteur étudié |
|---|---|---|
| B0 | TF-IDF + nettoyage | baseline |
| F1 | Sentence-BERT + nettoyage | système complet |
| A1 | Sentence-BERT **sans** nettoyage | apport du nettoyage |

$\Delta = P@3(\text{F1}) - P@3(\text{config})$ : F1−B0 = apport de SBERT, F1−A1 = apport du nettoyage.
Un delta nul ou négatif signifie **aucun bénéfice démontré sur ce test**.

In [ ]:
res = S.results_dir
ml_path = res / "ablation_ml_summary.csv"
PROXY = not ml_path.exists()
if PROXY:
    ml_path = res / "ablation_ml_summary_PROXY.csv"
    display(Markdown("> ⚠ Résultats obtenus avec des **labels proxy** : test technique uniquement."))
ml = pd.read_csv(ml_path)
display(ml.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
colors = ["#8C8C8C", "#4C72B0", "#DD8452"][: len(ml)]
ax[0].bar(ml.config, ml.mean_p_at_k, color=colors); ax[0].set_ylim(0, 1.05)
ax[0].set_title("Precision@3 moyenne"); ax[0].plot(ml.config, ml.mean_p_at_k_max, "k_", ms=40,
                                                   label="maximum atteignable"); ax[0].legend()
for i, v in enumerate(ml.mean_p_at_k):
    ax[0].text(i, v + 0.02, f"{v:.2f}", ha="center")
ax[1].bar(ml.config, ml.median_latency_ms, color=colors); ax[1].set_title("Latence médiane du classement (ms)")
plt.tight_layout(); plt.show()

if "delta_vs_F1" in ml:
    g = ml.set_index("config").mean_p_at_k
    display(Markdown(f"- **Apport de SBERT** (F1 − B0) : **{g['F1_sbert_clean'] - g['B0_tfidf']:+.3f}**\n"
                     f"- **Apport du nettoyage** (F1 − A1) : **{g['F1_sbert_clean'] - g['A1_sbert_noclean']:+.3f}**"))

In [ ]:
# Détail par CV : où les modèles diffèrent-ils ?
per_cv = pd.read_csv(res / ("ablation_ml_PROXY.csv" if PROXY else "ablation_ml.csv"))
pivot = per_cv.pivot(index="cv_id", columns="config", values="p_at_k")
display(pivot.round(2).T)
pivot.plot.bar(figsize=(14, 3.5), width=0.8, title="Precision@3 par CV"); plt.ylim(0, 1.05)
plt.legend(ncol=3); plt.tight_layout(); plt.show()

### 8.2 Ablation des explications (Q3)

| Test | Configuration |
|---|---|
| L0 | explication à règles |
| L1 | Gemini avec la description de l'offre |
| A2 | Gemini **sans** la description de l'offre (titre seul) |

Mêmes CV, **même Top 3 figé** pour les trois configurations. Mesures : affirmations non retrouvées dans les sources
(automatique, via le garde-fou), latence, et grille manuelle à deux (`manual_evidence_ok`, `manual_invented`,
`manual_utility_1_3`).

In [ ]:
from cvmatch.evaluation.ablation_llm import summarize_llm
llm = pd.read_csv(res / "ablation_llm.csv")
display(summarize_llm(llm.copy()))
if llm.config.nunique() == 1:
    display(Markdown("> ℹ Seule la configuration L0 a été évaluée (pas de clé Gemini lors de l'exécution)."))
display(llm[["config", "cv_id", "job_title", "unsupported_claims_auto", "explanation"]].head(6))

### 8.3 Analyse des résultats

> *À rédiger à partir des tableaux ci-dessus, après annotation réelle :*
> - **Q1** — SBERT vs TF-IDF : valeur et signe du delta, cas où SBERT aide (synonymes FR/EN), cas où il échoue.
> - **Q2** — effet du nettoyage sur SBERT.
> - **Q3** — nombre d'affirmations retirées par le garde-fou (L1 vs A2), utilité perçue vs règles, coût en latence/quota.
> - Taille de l'échantillon et accord inter-annotateurs : quelle confiance accorder aux écarts observés ?

## 9. Démonstration du système complet

Le même objet `MatchingService` est utilisé par l'API FastAPI (`cvmatch serve`), l'interface Streamlit
(`ui/app.py`), la ligne de commande et ce notebook. Ici : CV au format **PDF** → Top 3 + explications.

In [ ]:
from cvmatch.service import build_service
from cvmatch.preprocessing.pdf import extract_pdf_text

if "Sentence-BERT (F1)" not in rankers:   # embeddings absents : on reste sur TF-IDF
    S.ranker = "tfidf"
svc = build_service(S)
pdfs = sorted(S.cvs_dir.glob("*.pdf"))
cv_text = extract_pdf_text(pdfs[0]) if pdfs else raw_cv
out = svc.match(cv_text, top_k=3, explainer="gemini" if S.gemini_api_key else "rules")

display(Markdown(f"**CV :** `{pdfs[0].name if pdfs else cv_id}` · **ranker :** {out.ranker} · "
                 f"**explications :** {out.explainer} · **temps total :** {out.timings_ms['total']:.0f} ms"))
display(Markdown("**Compétences détectées :** " + (", ".join(out.resume_skills) or "—")))
for i, r in enumerate(out.recommendations, 1):
    e = r.explanation
    display(Markdown(
        f"#### {i}. {r.job_title}  —  similarité {r.similarity_score:.2f}\n"
        f"*{' · '.join(x for x in [r.company, r.city, r.sector] if x)}*\n\n"
        f"{e.explanation}\n\n"
        f"✅ **En commun :** {', '.join(e.matching_skills) or '—'}  \n"
        f"📈 **À développer :** {', '.join(e.missing_skills) or '—'}"))
for w in out.warnings:
    print("⚠", w)

## 10. Éthique et vie privée

- **Minimisation des données.** E-mail, téléphone, liens personnels et adresse sont masqués *avant* toute
  journalisation ou appel à un service externe ; les CV ne sont jamais stockés par l'API.
- **Données de test.** Uniquement des CV fictifs, anonymisés ou consentis ; les CV sont exclus du dépôt Git
  (`.gitignore`) et de l'archive du projet.
- **Transparence.** Le score est présenté comme une proximité textuelle, **jamais** comme une probabilité d'embauche ;
  chaque recommandation est expliquée et justifiée par des extraits.
- **Hallucinations.** Le garde-fou supprime toute compétence ou preuve absente des sources, et le système reste
  fonctionnel sans LLM (repli sur les règles).
- **Biais.** Le catalogue reflète le marché Rekrute 2019-2022 (surreprésentation de certains secteurs et villes) ;
  le système ne filtre pas sur des critères sensibles (âge, genre, nationalité) et n'en utilise aucun.
- **Secrets.** La clé Gemini est lue depuis `.env` / secrets Colab, jamais écrite dans le code ni le notebook.

## 11. Limites et perspectives

**Limites**
- Petit jeu de test annoté ; **biais de pooling** (le pool provient des modèles évalués), atténué mais non supprimé
  par les négatifs aléatoires et la double annotation.
- Catalogue daté (2005-2022, récent = 2019-2022) ; pas de filtre d'éligibilité (diplôme, années d'expérience, ville).
- Dictionnaire de compétences limité (~100 entrées) ; SBERT tronque les très longues offres.
- Explications Gemini dépendantes d'un service externe (quota, latence, coût).

**Perspectives**
- Ré-ordonnancement par *cross-encoder* sur le Top 20 ; *fine-tuning* SBERT sur des paires annotées.
- Filtres d'éligibilité (niveau d'études, expérience, localisation) et préférences de l'utilisateur.
- Extraction de compétences apprise (NER) au lieu d'un dictionnaire.
- Catalogue actualisé en continu ; évaluation utilisateur (étude qualitative).

## 12. Conclusion

Nous avons conçu un système complet et reproductible qui, à partir d'un CV PDF, recommande les trois offres
d'emploi marocaines les plus proches et explique chaque recommandation. Le projet combine une baseline lexicale
(TF-IDF), un modèle sémantique multilingue (Sentence-BERT) et un LLM (Gemini) encadré par un garde-fou de fidélité,
le tout évalué par un protocole d'ablation rigoureux (pool figé, double annotation, un seul facteur modifié à la
fois). L'architecture modulaire (pipeline par étapes, API, interface, tests automatisés) permet de reproduire
l'ensemble des résultats par une seule commande.

> *Compléter avec les principaux chiffres mesurés (section 8) et la réponse à Q1, Q2, Q3.*

## 13. Annexes

### A. Architecture du code

In [ ]:
def tree(path, prefix="", depth=0, max_depth=3):
    skip = {"__pycache__", ".venv", ".git", "data", "artifacts", "results", "dist", ".ipynb_checkpoints",
            ".pytest_cache", ".ruff_cache"}
    items = sorted([p for p in path.iterdir() if p.name not in skip and not p.name.endswith(".egg-info")
                    and not (p.name.startswith(".") and p.name not in {".env.example", ".vscode"})],
                   key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(items):
        last = i == len(items) - 1
        print(prefix + ("└── " if last else "├── ") + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir() and depth < max_depth - 1:
            tree(p, prefix + ("    " if last else "│   "), depth + 1, max_depth)
print(ROOT.name + "/"); tree(ROOT)

### B. Reproduire les résultats

```bash
pip install -e ".[all]"              # installation
cp .env.example .env                 # (optionnel) GEMINI_API_KEY=...
cvmatch pipeline                     # prepare → embed → cvs → pool → labels → ablate-ml → ablate-llm → report
# annoter data/eval/pool_to_annotate.csv (annotator_1, annotator_2), puis :
cvmatch pipeline --from labels
cvmatch serve  &  streamlit run ui/app.py      # application
pytest                               # tests automatisés
```

### C. Traçabilité de l'exécution

In [ ]:
import json
manifest = json.loads((S.results_dir / "pipeline_run.json").read_text(encoding="utf-8"))
display(pd.DataFrame([{"étape": k, "statut": v.get("status"), "durée (s)": v.get("seconds")}
                      for k, v in manifest["steps"].items()]))
print("Configuration utilisée :")
print(json.dumps(manifest["config"], indent=2, ensure_ascii=False))

In [ ]:
# Tests automatisés du projet
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", "--color=no", "-p", "no:cacheprovider"],
                   cwd=ROOT, capture_output=True, text=True)
print((r.stdout or r.stderr).strip().splitlines()[-1])